# 🔨 Step 1b — Compile llama.cpp & Save as Kaggle Dataset

This notebook compiles `llama.cpp` from source with CUDA support and saves the binaries as a **permanent private Kaggle Dataset**.

Run this notebook:
- **Once** to create the dataset
- **Again** whenever you want to update llama.cpp to a newer version

The server notebook (`2_run_server.ipynb`) will then mount this dataset and copy everything in seconds — no more 26-minute compilation on every session.

---

### What gets saved

| File | Purpose |
|---|---|
| `llama-server` | The OpenAI-compatible HTTP server exposed via Cloudflare |
| `llama-cli` | Interactive CLI for quick local tests |
| `llama-mtmd-cli` | Multimodal CLI (text + image) |
| `lib/*.so*` | Shared libraries required by the binaries at runtime |
| `version.json` | Commit hash + build info — useful to track which version is running |

> ⚠️ The `.so` files are essential. Without them, `llama-server` crashes at startup
> with `cannot open shared object file` even if the binary itself is present.

---

### ⚠️ Prerequisites

- [ ] **Accelerator** → **GPU T4 x1** (needed for CUDA compilation)
- [ ] **Internet** → **ON**
- [ ] **Persistence** → not needed for this notebook

## ⚙️ Cell 1 — Configuration

In [ ]:
import os
from pathlib import Path

# ── Your Kaggle username ───────────────────────────────────────────────────────
# Find it at: https://www.kaggle.com/settings (under 'Username')
KAGGLE_USERNAME = ""   # ← e.g. "johndoe"

# ── Dataset slug that will be created on your Kaggle account ──────────────────
DATASET_SLUG = "llama-cpp-bin"

# ── Directories ───────────────────────────────────────────────────────────────
# We use /tmp to avoid the 20 GB /kaggle/working quota
BUILD_DIR  = Path("/tmp/llama_build")    # compilation happens here
OUTPUT_DIR = Path("/tmp/llama_output")   # binaries + .so files go here

# ══════════════════════════════════════════════════════════════════════════════

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
if not KAGGLE_USERNAME:
    raise ValueError("Please set KAGGLE_USERNAME at the top of this cell.")

BUILD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Config ready")
print(f"   Kaggle username : {KAGGLE_USERNAME}")
print(f"   Dataset slug    : {DATASET_SLUG}")
print(f"   Build dir       : {BUILD_DIR}")
print(f"   Output dir      : {OUTPUT_DIR}")

## 🔨 Cell 2 — Compile llama.cpp

This is the only long step (~26 minutes). It:
1. Installs build dependencies
2. Applies the Kaggle CUDA fix (`libcuda.so` symlink)
3. Clones and compiles llama.cpp with CUDA support for the T4 (sm_75)

> ⏳ Expected time: **~26 minutes** — but only ever once (or when you choose to update).

In [ ]:
%%bash
set -e

# ── System packages ────────────────────────────────────────────────────────────
echo "📦 Installing build dependencies..."
apt-get update -qq
apt-get install -qq pciutils build-essential cmake curl libcurl4-openssl-dev

# ── CUDA fix for Kaggle ────────────────────────────────────────────────────────
# libcuda.so lives in /usr/local/nvidia/lib64 (the real GPU driver mount).
# cmake looks for it in /usr/local/cuda/lib64 — we bridge with a symlink.
echo "🔧 Applying CUDA symlink fix..."
ln -sf /usr/local/nvidia/lib64/libcuda.so /usr/local/cuda/lib64/libcuda.so 2>/dev/null || true

# ── Clone llama.cpp ────────────────────────────────────────────────────────────
echo "📥 Cloning llama.cpp..."
rm -rf /tmp/llama_build/llama.cpp
git clone --depth=1 https://github.com/ggml-org/llama.cpp /tmp/llama_build/llama.cpp

cd /tmp/llama_build/llama.cpp
COMMIT=$(git rev-parse --short HEAD)
DATE=$(git log -1 --format='%ci')
echo "   Commit : $COMMIT  ($DATE)"
echo "$COMMIT" > /tmp/llama_build/commit.txt
echo "$DATE"   >> /tmp/llama_build/commit.txt

# ── Configure ─────────────────────────────────────────────────────────────────
echo "⚙️  Configuring build (CUDA sm_75 = T4)..."
cmake -B build \
    -DGGML_CUDA=ON \
    -DGGML_NATIVE=OFF \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES=75 \
    -DCMAKE_PREFIX_PATH=/usr/local/nvidia \
    2>&1 | tail -3

# ── Compile ───────────────────────────────────────────────────────────────────
echo "🔨 Compiling (~26 minutes)..."
cmake --build build -j$(nproc) 2>&1 | tail -5

echo ""
echo "✅ Compilation complete."
echo "Binaries:"
ls -lh build/bin/llama-server build/bin/llama-cli build/bin/llama-mtmd-cli
echo "Shared libraries:"
ls -lh build/lib/*.so* 2>/dev/null || echo "   (no .so files in build/lib — checking src)"
find build -name '*.so*' -not -path '*/CMakeFiles/*' 2>/dev/null | head -20

## 📋 Cell 3 — Collect binaries + shared libraries

We copy both the executables **and** the `.so` shared libraries they depend on.

This is critical: `llama-server` is dynamically linked against `libllama-common.so` and others.
If only the binary is copied, it crashes at startup with `cannot open shared object file`.

In [ ]:
import shutil, subprocess, json, os
from pathlib import Path

BIN_DIR   = Path("/tmp/llama_build/llama.cpp/build/bin")
BUILD_ROOT = Path("/tmp/llama_build/llama.cpp/build")

# ── Clean output dir ──────────────────────────────────────────────────────────
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

# ── Copy binaries ─────────────────────────────────────────────────────────────
binaries = ["llama-server", "llama-cli", "llama-mtmd-cli"]
for name in binaries:
    src = BIN_DIR / name
    assert src.exists(), f"Binary not found: {src}"
    dst = OUTPUT_DIR / name
    shutil.copy2(src, dst)
    dst.chmod(0o755)

# ── Copy ALL shared libraries found in the build tree ─────────────────────────
# llama-server links against libllama-common.so and potentially others.
# We collect every .so file from the build to be safe.
so_files = [
    f for f in BUILD_ROOT.rglob("*.so*")
    if "CMakeFiles" not in str(f) and f.is_file()
]
for so in so_files:
    shutil.copy2(so, OUTPUT_DIR / so.name)

# ── Set LD_LIBRARY_PATH so version check works ────────────────────────────────
os.environ["LD_LIBRARY_PATH"] = str(OUTPUT_DIR)

# ── Write version.json ────────────────────────────────────────────────────────
commit_lines = Path("/tmp/llama_build/commit.txt").read_text().strip().splitlines()
commit_hash  = commit_lines[0]
commit_date  = commit_lines[1] if len(commit_lines) > 1 else "unknown"

ver = subprocess.run(
    [str(OUTPUT_DIR / "llama-server"), "--version"],
    capture_output=True, text=True,
    env={**os.environ, "LD_LIBRARY_PATH": str(OUTPUT_DIR)}
)
version_str = (ver.stdout or ver.stderr).strip().splitlines()[0] if (ver.stdout or ver.stderr) else "unknown"

version_info = {
    "llama_cpp_commit" : commit_hash,
    "commit_date"      : commit_date,
    "llama_server_ver" : version_str,
    "cuda_arch"        : "sm_75 (T4)",
    "built_on"         : "Kaggle",
    "ld_library_path"  : "/kaggle/working",
}
(OUTPUT_DIR / "version.json").write_text(json.dumps(version_info, indent=2))

# ── Summary ───────────────────────────────────────────────────────────────────
print("📦 Output directory contents:")
print()
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size / (1024**2)
    tag  = "← binary" if f.name in binaries else ("← shared lib" if ".so" in f.name else "")
    print(f"   {f.name:<35} {size:6.1f} MB  {tag}")

print()
print("📋 Version info:")
for k, v in version_info.items():
    print(f"   {k:<22} : {v}")

## 📦 Cell 4 — Create or update the Kaggle Dataset

If the dataset already exists (you're updating llama.cpp), we create a new version automatically.
If it doesn't exist yet (first run), we create it fresh.

In [ ]:
import json, subprocess

# ── Write dataset-metadata.json ───────────────────────────────────────────────
metadata = {
    "title"    : "llama-cpp-bin",
    "id"       : f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses" : [{"name": "mit"}],
    "isPrivate": True,
}
(OUTPUT_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

# ── Check if dataset already exists ───────────────────────────────────────────
check = subprocess.run(
    ["kaggle", "datasets", "status", f"{KAGGLE_USERNAME}/{DATASET_SLUG}"],
    capture_output=True, text=True
)
dataset_exists = check.returncode == 0

if dataset_exists:
    print("🔄 Dataset already exists — creating a new version...")
    result = subprocess.run(
        ["kaggle", "datasets", "version",
         "-p", str(OUTPUT_DIR),
         "-m", f"llama.cpp update: {version_info['llama_cpp_commit']}",
         "--dir-mode", "skip"],
        capture_output=True, text=True
    )
else:
    print(f"✨ Creating new dataset: {KAGGLE_USERNAME}/{DATASET_SLUG}")
    result = subprocess.run(
        ["kaggle", "datasets", "create",
         "-p", str(OUTPUT_DIR),
         "--dir-mode", "skip"],
        capture_output=True, text=True
    )

print(result.stdout)
if result.returncode != 0:
    print("⚠️ stderr:", result.stderr)
else:
    print(f"\n✅ Dataset ready: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")

## ✅ Done!

Your llama.cpp binaries and shared libraries are now stored permanently in your Kaggle account.

### What's next?

In `2_run_server.ipynb`, add this dataset as an input:
1. Kaggle notebook settings → **Add data** → search for `llama-cpp-bin`
2. Files will be available at `/kaggle/input/datasets/{username}/llama-cpp-bin/`

### To update llama.cpp in the future

Simply re-run this notebook. Cell 4 detects that the dataset already exists and creates a new version automatically.